# Understanding Customer Intent Classification

This notebook implements the mandatory part: reproducible preprocessing, RNN and LSTM intent classifiers, validation-based model selection, `Fallback Intent` handling, pickle serialization, and `intents.csv` submission generation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from intent_classifier import load_data, train

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT
train_frame, test_frame, text_column, label_column = load_data(DATA_DIR)
print(f'train rows: {len(train_frame)}, test rows: {len(test_frame)}')
print(f'text column: {text_column}, label column: {label_column}')
print(f'intents: {train_frame.label.nunique()}')
train_frame.head()

## RNN and LSTM comparison

The training function uses a stratified, deterministic validation split. Both architectures use the same vocabulary and padded sequences; the model with the highest validation accuracy is serialized.

In [ ]:
train(DATA_DIR, OUTPUT_DIR, epochs=25, batch_size=32)

## Submission checks

`model.pkl` stores the selected architecture, vocabulary, labels, weights, sequence length, and calibrated fallback threshold. `intents.csv` contains one `intent` column in test-row order.

In [ ]:
import pickle
import pandas as pd

with (OUTPUT_DIR / 'model.pkl').open('rb') as handle:
    artifact = pickle.load(handle)
submission = pd.read_csv(OUTPUT_DIR / 'intents.csv')
print('architecture:', artifact.architecture)
print('fallback threshold:', artifact.fallback_threshold)
print('submission shape:', submission.shape)
assert len(submission) == len(test_frame)
assert list(submission.columns) == ['intent']
submission.head()

## Optional BERT experiment

This cell documents the required transfer-learning route. It is intentionally guarded: it runs when `transformers` is installed and the pretrained checkpoint is available, but does not prevent the CPU-safe RNN/LSTM submission from running offline.

In [ ]:
try:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from torch.utils.data import Dataset

    class IntentDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_length=128):
            self.tokens = tokenizer(texts, truncation=True, padding=True, max_length=max_length)
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            item = {key: torch.tensor(value[index]) for key, value in self.tokens.items()}
            item['labels'] = torch.tensor(self.labels[index])
            return item

    import torch
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    bert_texts_train, bert_texts_valid, bert_labels_train, bert_labels_valid = train_test_split(
        train_frame['text'].tolist(),
        train_frame['label'].map({label: index for index, label in enumerate(sorted(train_frame['label'].unique()))}).tolist(),
        test_size=0.2,
        random_state=42,
        stratify=train_frame['label'],
    )
    checkpoint = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    bert_train = IntentDataset(bert_texts_train, bert_labels_train, tokenizer)
    bert_valid = IntentDataset(bert_texts_valid, bert_labels_valid, tokenizer)
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=train_frame['label'].nunique(),
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    bert_model.to(device)
    optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)
    loader = torch.utils.data.DataLoader(bert_train, batch_size=16, shuffle=True)
    for epoch in range(2):
        bert_model.train()
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad()
            loss = bert_model(**batch).loss
            loss.backward()
            optimizer.step()
    bert_model.eval()
    valid_loader = torch.utils.data.DataLoader(bert_valid, batch_size=32)
    predictions = []
    with torch.no_grad():
        for batch in valid_loader:
            inputs = {key: value.to(device) for key, value in batch.items() if key != 'labels'}
            predictions.extend(bert_model(**inputs).logits.argmax(dim=1).cpu().tolist())
    print('BERT validation accuracy:', accuracy_score(bert_labels_valid, predictions))
except (ImportError, OSError) as error:
    print('Optional BERT experiment skipped:', error)
    print('Install transformers and allow the checkpoint download to run it.')